# Домашняя работа 2 — качество данных

## Подготовка

In [4]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
LOCAL = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
print("pandas", pd.__version__)

pandas 2.2.2


## Независимая загрузка

In [5]:
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
export = pd.read_csv(LOCAL / "training_export.csv", dtype="string")
raw_copy = export.copy(deep=True)
print(decisions.shape, export.shape)
print(export.dtypes)
export.head(5)

(6830, 7) (6832, 7)
source_row       string[python]
activity_year    string[python]
lei              string[python]
county_code      string[python]
action_taken     string[python]
loan_amount      string[python]
income           string[python]
dtype: object


,source_row,activity_year,lei,county_code,action_taken,loan_amount,income
0,2,2023,549300VZVN841I2ILS84,11001,1,oops,72.0
1,3,2023,549300P1347BFCBHJV60,11001,88,585000.0,174.0
2,5,2023,549300P1347BFCBHJV60,11001,1,275000.0,BAD
3,7,2023,549300RPOGWJRH63HS39,11001,1,295000.0,63.0
4,8,2023,549300RPOGWJRH63HS39,11001,1,1125000.0,247.0


## H1. Собственная функция очистки

In [6]:
def clean_export(source):
    df = source.copy(deep=True)
    audit = {"input": len(df)}

    known = df["source_row"].notna()
    versions = df[known].drop_duplicates()
    conflicts = int(versions["source_row"].duplicated().sum())
    assert conflicts == 0, f"Конфликтующих версий source_row: {conflicts}"

    is_replay = known & df.duplicated(keep="first")
    audit["replays"] = int(is_replay.sum())
    df = df[~is_replay].copy()

    original = df.copy()
    df["county_code"] = df["county_code"].str.strip()

    parse_errors = 0
    for col in ["source_row", "action_taken", "loan_amount", "income"]:
        converted = pd.to_numeric(df[col], errors="coerce")
        parse_errors += int((df[col].notna() & converted.isna()).sum())
        df[col] = converted
    audit["parse_errors"] = parse_errors

    rules = {
        "source_row неизвестен": df["source_row"].notna(),
        "исход вне 1-3": df["action_taken"].isin([1, 2, 3]),
        "сумма неизвестна или не положительна": df["loan_amount"].gt(0),
        "территория не 11001": df["county_code"].eq("11001"),
    }
    reason = pd.Series("", index=df.index, dtype="object")
    for text, ok in rules.items():
        ok = ok.fillna(False).astype(bool)
        reason = reason.mask((reason == "") & ~ok, text)
    passed = reason == ""

    clean = df[passed].copy()
    quarantine = original[~passed].copy()
    quarantine["reason"] = reason[~passed]

    audit["clean"] = len(clean)
    audit["quarantine"] = len(quarantine)
    assert audit["input"] == audit["replays"] + audit["clean"] + audit["quarantine"]
    return clean, quarantine, audit

## H2. Год и известный доход

In [7]:
clean, quarantine, audit = clean_export(export)
print(audit)

assert clean["activity_year"].eq("2023").fillna(False).all()

total = len(clean)
income_known = int(clean["income"].notna().sum())
income_unknown = int(clean["income"].isna().sum())
mean_income = clean["income"].mean()

print("Подготовленных строк:", total)
print("Доход известен:", income_known, "| неизвестен:", income_unknown)
print(f"Средний известный доход: {mean_income:.2f} тыс. USD в год")

assert income_known + income_unknown == total
assert audit["input"] == audit["replays"] + audit["clean"] + audit["quarantine"]
assert_frame_equal(export, raw_copy)   # вход не изменён

{'input': 6832, 'replays': 2, 'parse_errors': 2, 'clean': 6828, 'quarantine': 2}
Подготовленных строк: 6828
Доход известен: 6485 | неизвестен: 343
Средний известный доход: 237.07 тыс. USD в год


## H3. Проверка отчётного года

In [8]:
wrong_year = export.copy(deep=True)
wrong_year.loc[wrong_year.index[0], "activity_year"] = "2022"

year_ok = wrong_year["activity_year"].eq("2023").fillna(False).all()
print("year_ok:", year_ok)
assert not year_ok
assert_frame_equal(export, raw_copy)

year_ok: False


## H+1. Конфликт исходных значений

In [9]:
conflict = export.copy(deep=True)
replay_idx = export.index[export.duplicated(keep="first") & export["source_row"].notna()]
last = replay_idx[-1]
assert conflict.loc[last, "loan_amount"] != "999999"
conflict.loc[last, "loan_amount"] = "999999"

try:
    clean_export(conflict)
except AssertionError as e:
    print("Ожидаемая остановка:", e)
else:
    print("Ошибка проверки: конфликт не обнаружен")

Ожидаемая остановка: Конфликтующих версий source_row: 1


## H+2. Совпадение опубликованных полей

In [10]:
cols = [c for c in decisions.columns if c != "source_row"]
dedup = decisions.drop_duplicates(subset=cols)

print("Строк до:", len(decisions), "| после:", len(dedup),
      "| разница:", len(decisions) - len(dedup))
print("Сумма loan_amount до:", decisions["loan_amount"].sum(),
      "| после:", dedup["loan_amount"].sum(),
      "| разница:", decisions["loan_amount"].sum() - dedup["loan_amount"].sum())

Строк до: 6830 | после: 6757 | разница: 73
Сумма loan_amount до: 4577460000.0 | после: 4547855000.0 | разница: 29605000.0
